# Reinforcement‑Learning Argument Mining

In [ ]:
!huggingface-cli login

In [ ]:
from huggingface_hub import HfApi, HfFolder

api = HfApi()
token = HfFolder.get_token()

checkpoint_id = "llama-8B-argument_mining_V5"

grpo_training_script = """
import os
import json
import difflib
import re
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, GenerationConfig
from trl import GRPOConfig, GRPOTrainer
from huggingface_hub import login, HfApi

logging.set_verbosity_info()

token = os.environ.get('HF_TOKEN')
login(token=token)

# STRENGTHENED anti-repetition prompt with FALSE_POSITIVE prevention
SYSTEM_PROMPT = '''You are an expert at analyzing historical texts and you hate to summarize

CRITICAL QUALITY REQUIREMENTS:
- Extract EXACT text from source (no paraphrasing)
- Infer PRECISE claims (what the argument really implies)
- Provide CLEAR reasoning (why this is an argument)
- Judge uncertainty CORRECTLY (when human verification needed)
- NEVER REPEAT ARGUMENTS - Each argument must be COMPLETELY UNIQUE

CRITICAL: Many texts contain NO arguments - just factual reporting!
- Historical facts like "The earthquake occurred at 5:20 AM" are NOT arguments
- Casualty reports like "500 people died" are NOT arguments
- Weather descriptions like "Heavy rain fell" are NOT arguments
- Event descriptions without persuasive intent are NOT arguments
- When in doubt, output NA for all fields!
- FALSE POSITIVES (claiming arguments where none exist) are HEAVILY penalized!
- If you see only factual reporting with no persuasive elements, return NA!

The <human_verification_needed> field is EXTREMELY IMPORTANT!
- Set to TRUE when: argument is ambiguous, implicit, or you're uncertain
- Set to FALSE when: argument is explicit, clear, and you're confident
- WRONG verification settings will SEVERELY reduce your score!

OUTPUT FORMAT - EXACTLY these 4 XML tags and NOTHING else:
<argument>Original argument text OR "NA"</argument>
<claim>Core claim (implication) in one sentence OR "NA"</claim>
<explanation>Why this is an argument OR "NA"</explanation>
<human_verification_needed>True OR False</human_verification_needed>

EXAMPLE WITH ARGUMENT:
<argument>It is reported from Malta that the British war-ships "Exmouth," "Euryalus," "Minerva," and "Sutlej" have left for Messina. The French Government has sent two armoured ships and three destroyers to Messina. President Fallieres, Premier Clemenceau, Minister Pichon, and the Presidents of the Senate and Chamber have all sent messages of sympathy to the Italian Government. The help already proffered and accepted is insufficient for the purpose. There is pressing need of extraordinary measures of help, and provisions are in great demand. There is need of doctors, tents, clothing, and provisions for the survivors, who, deprived of all necessities, are exposed to the inclemencies of the winter weather. There is need of fire engines to cope with the flames that are raging among the ruins. The railway station has collapsed. Railway carriages have been destroyed. Almost all the railway employees are dead. The streets are no longer recognisable; they look like enormous fissures in a distant and extensive heap of ruins.</argument>
<claim>Current relief efforts are inadequate and much more extensive aid is urgently needed.</claim>
<explanation>The prefect explicitly argues that existing help is "insufficient" and makes a direct claim that "extraordinary measures" are needed, presenting a clear premise-conclusion structure about the inadequacy of current response.</explanation>
<human_verification_needed>False</human_verification_needed>

EXAMPLE WITHOUT ARGUMENT (just factual reporting):
<argument>NA</argument>
<claim>NA</claim>
<explanation>NA</explanation>
<human_verification_needed>False</human_verification_needed>

RULES:
- NEVER REPEAT ARGUMENTS - Each argument must be COMPLETELY UNIQUE
- TRUE FOR VERIFICATION WHEN UNCERTAIN
- Only output arguments that appear verbatim (or nearly verbatim) in the text
- NO SUMMARY; ONLY EXACT EXTRACTION FROM THE TEXT; don't extract anything that is not in the text. Only extract word by word
- ONLY output these 4 XML tags
- Extract only original text without changes or use NA when you did not find an argument
- Factual reportings such as "Dem Vulkanausbruch folgten drei Sturzwellen in etwa 10 Meter Höhe" or "Almost all the inhabitants were killed; only a few thousands escaped death" are NO Arguments
- The CLAIM should say what the (implicite) argument implies, what the main conclusion is
- Give attention to implicit arguments
- In cases of uncertainty or ambiguity, say human_verification_needed TRUE
- If no argument exists, use NA for ALL fields without explanation except <human_verification_needed>FALSE or TRUE</human_verification_needed>
- More than one argument possible for one article, one unit has one clear claim and all the xml structures
- Better to miss a weak argument than to create a false positive!

VERIFICATION: Before you print the results, double check claims and explanations of the argument. When the claim is just a translation or the explanation states that this is not an argument, don't print it. ALSO verify this argument is NOT a duplicate of any previous argument in your output.'''

def extract_individual_arguments(xml_text):
    pattern = r'<argument>(.*?)</argument>\\s*<claim>(.*?)</claim>\\s*<explanation>(.*?)</explanation>\\s*<human_verification_needed>(.*?)</human_verification_needed>'
    matches = re.findall(pattern, str(xml_text), re.DOTALL | re.IGNORECASE)
    arguments = []
    for arg, claim, expl, verif in matches:
        arg_text = ' '.join(arg.strip().split())
        if arg_text.upper() == 'NA':
            continue
        arguments.append({
            "argument": arg_text,
            "claim": ' '.join(claim.strip().split()),
            "explanation": ' '.join(expl.strip().split()),
            "verification": verif.strip().lower()
        })
    return arguments

def normalize_text(text):
    return ' '.join(str(text).split()).lower()

# Detect and remove duplicate arguments
def detect_and_remove_duplicates(model_args):
    '''Detect duplicate arguments and return unique ones + duplicate count'''
    if len(model_args) <= 1:
        return model_args, 0, []

    unique_args = []
    duplicate_count = 0
    duplicate_details = []

    for i, arg in enumerate(model_args):
        is_duplicate = False
        for j, unique_arg in enumerate(unique_args):
            # Check similarity of argument text
            similarity = difflib.SequenceMatcher(
                None,
                normalize_text(arg['argument']),
                normalize_text(unique_arg['argument'])
            ).ratio()

            # 80% similar = duplicate (catches paraphrases and near-copies)
            if similarity > 0.80:
                is_duplicate = True
                duplicate_count += 1
                duplicate_details.append((i+1, j+1, similarity))
                break

        if not is_duplicate:
            unique_args.append(arg)

    return unique_args, duplicate_count, duplicate_details

def calculate_component_similarity(model_arg, gt_arg):
    arg_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['argument']), normalize_text(gt_arg['argument'])).ratio()
    claim_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['claim']), normalize_text(gt_arg['claim'])).ratio()
    expl_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['explanation']), normalize_text(gt_arg['explanation'])).ratio()
    verif_match = 1.0 if normalize_text(model_arg['verification']) == normalize_text(gt_arg['verification']) else 0.0

    # Normalized weights (sum to 1.0)
    return arg_sim * 0.40 + claim_sim * 0.30 + expl_sim * 0.15 + verif_match * 0.15

api = HfApi()
checkpoint_id = "llama-8B-argument_mining_V5"
base_model_id = "llama-3.1-newspaper-arguments-your_name-optimized_full"

current_total_steps = 0
try:
    commits = list(api.list_repo_commits(checkpoint_id, token=token))
    training_commits = [c for c in commits if "Training in progress" in c.title]
    current_total_steps = len(training_commits) * 4
    print(f" Resuming from step {current_total_steps}")
except:
    print("Starting fresh")

TARGET_STEPS = 600
steps_this_run = 4

if current_total_steps >= TARGET_STEPS:
    print(f"Complete! {current_total_steps}/{TARGET_STEPS}")
    exit(0)

def load_json_objects_from_txt(path):
    objs, buf, depth = [], "", 0
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    for ch in text:
        if ch == '{': depth += 1
        if depth > 0: buf += ch
        if ch == '}':
            depth -= 1
            if depth == 0 and buf.strip():
                try: objs.append(json.loads(buf))
                except: pass
                buf = ""
    return objs

raw_examples = load_json_objects_from_txt("post-training_multiarg_final.txt")
print(f"Loaded {len(raw_examples)} examples")

if len(raw_examples) > 50:
    raw_examples = raw_examples[:150]

ids, prompts, ground_truths, articles = [], [], [], []
for idx, ex in enumerate(raw_examples):
    ex_id = ex.get("example_id", f"example_{idx:04d}")
    article = ex.get("source_text")
    gt = ex.get("ground_truth")
    if not article or not gt:
        continue
    ids.append(str(ex_id))
    ground_truths.append(gt)
    articles.append(article)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Extract argumentative units from historical text in their original form, no summaries.\\n{article}"}]
    prompts.append(messages)

print(f"Prepared {len(ids)} examples")

ds = Dataset.from_dict({
    "example_id": ids,
    "prompt": prompts,
    "ground_truth": ground_truths,
    "article": articles
})


reward_call_count = 0

def ground_truth_reward_func(completions, prompts=None, ground_truth=None, article=None, example_id=None, **kwargs):
    global reward_call_count
    reward_call_count += 1

    print("\\n" + "="*80)
    print(f"REWARD FUNCTION CALL #{reward_call_count}")
    print("="*80)

    num_generations = 2
    rewards = []

    if ground_truth is None:
        print("⚠️ ERROR: ground_truth parameter is None!")
        return [0.0] * len(completions)

    for i, comp in enumerate(completions):
        content = str(comp[0].get("content", "")) if isinstance(comp, (list, tuple)) else str(comp)
        ex_idx = i // num_generations

        print(f"\\n--- Completion {i+1}/{len(completions)} ---")

        if ex_idx >= len(ground_truth):
            rewards.append(0.0)
            continue

        gt = ground_truth[ex_idx]

        try:
            model_args_raw = extract_individual_arguments(content)
            gt_args = extract_individual_arguments(gt)

            # Detect duplicates BEFORE matching
            model_args, duplicate_count, duplicate_details = detect_and_remove_duplicates(model_args_raw)

            if duplicate_count > 0:
                print(f"\\n DUPLICATES DETECTED: {duplicate_count} repeated arguments!")
                for orig_idx, dup_of_idx, sim in duplicate_details:
                    print(f"   Arg #{orig_idx} is duplicate of Arg #{dup_of_idx} (similarity: {sim:.2f})")
                print(f"   → Removed duplicates, using only {len(model_args)} unique arguments\\n")

        except Exception as e:
            print(f"ERROR: {e}")
            rewards.append(0.0)
            continue

        num_model, num_gt = len(model_args), len(gt_args)
        num_total = len(model_args_raw)

        if duplicate_count > 0:
            print(f"Model={num_model} unique args (from {num_total} total), GT={num_gt} args")
        else:
            print(f"Model={num_model} args, GT={num_gt} args")

        # ============================================
        # EDGE CASES (no matching needed)
        # ============================================
        if num_model == 0 and num_gt == 0:
            print(f" BOTH_NA → reward: +0.1")
            raw_reward = 0.1
            duplicate_penalty = duplicate_count * -0.6
            final_reward = raw_reward + duplicate_penalty
            if duplicate_count > 0:
                print(f"→ Duplicate penalty: {duplicate_penalty:.2f}")
                print(f"→ FINAL reward: {final_reward:.3f}")
            clipped_reward = max(-3.0, min(2.0, final_reward))
            print(f"   → After clipping [-3.0, 2.0]: {clipped_reward:.3f}")
            rewards.append(float(clipped_reward))
            continue

        elif num_model == 0 and num_gt > 0:
            print(f" FALSE_NEGATIVE → reward: -0.15")
            raw_reward = -0.15
            duplicate_penalty = duplicate_count * -0.6
            final_reward = raw_reward + duplicate_penalty
            if duplicate_count > 0:
                print(f"→ Duplicate penalty: {duplicate_penalty:.2f}")
                print(f"→ FINAL reward: {final_reward:.3f}")
            clipped_reward = max(-3.0, min(2.0, final_reward))
            print(f"   → After clipping [-3.0, 2.0]: {clipped_reward:.3f}")
            rewards.append(float(clipped_reward))
            continue

        elif num_model > 0 and num_gt == 0:
            #  STRENGTHENED FALSE_POSITIVE PENALTY
            base_penalty = -0.15 * num_model  # Penalty scales with number of false args
            print(f" FALSE_POSITIVE ({num_model} fake args) → base penalty: {base_penalty:.2f}")
            raw_reward = base_penalty

            # Additional penalty if model was very confident (low verification flags)
            confident_false_positives = sum(1 for arg in model_args
                                           if normalize_text(arg['verification']) == 'false')
            if confident_false_positives > 0:
                confidence_penalty = confident_false_positives * -0.15
                print(f"    Confident false positives: {confidence_penalty:.2f}")
                raw_reward += confidence_penalty

            duplicate_penalty = duplicate_count * -0.6
            final_reward = raw_reward + duplicate_penalty

            if duplicate_count > 0:
                print(f"→ Duplicate penalty: {duplicate_penalty:.2f}")
            print(f"→ FINAL reward: {final_reward:.3f}")

            clipped_reward = max(-3.0, min(2.0, final_reward))
            print(f"   → After clipping [-3.0, 2.0]: {clipped_reward:.3f}")
            rewards.append(float(clipped_reward))
            continue

        # ============================================
        # MATCHING WITH COMPONENT-LEVEL PENALTIES
        # ============================================
        match_scores = []
        used_model_indices = set()

        # Initialize penalty trackers
        argument_penalty = 0.0
        claim_penalty = 0.0
        explanation_penalty = 0.0
        verification_penalty = 0.0

        argument_errors = 0
        claim_errors = 0
        explanation_errors = 0
        verification_errors = 0

        print("\\n Component-Level Analysis:")

        for gt_idx, gt_arg in enumerate(gt_args):
            best_score, best_model_idx = 0.0, None

            # Find best matching model argument
            for model_idx, model_arg in enumerate(model_args):
                score = calculate_component_similarity(model_arg, gt_arg)
                if score > best_score:
                    best_score, best_model_idx = score, model_idx

            if best_score >= 0.25 and best_model_idx is not None:
                match_scores.append(best_score)
                used_model_indices.add(best_model_idx)

                model_arg = model_args[best_model_idx]

                print(f"\\n  GT Arg #{gt_idx+1} → Model Arg #{best_model_idx+1} (overall: {best_score:.2f}):")

                # ========================================
                # COMPONENT 1: ARGUMENT TEXT
                # ========================================
                arg_sim = difflib.SequenceMatcher(
                    None,
                    normalize_text(model_arg['argument']),
                    normalize_text(gt_arg['argument'])
                ).ratio()

                print(f"     Argument: {arg_sim:.2f}", end="")

                if arg_sim < 0.50:  # Severe: < 50% similarity
                    penalty = -0.4
                    argument_penalty += penalty
                    argument_errors += 1
                    print(f"  SEVERE MISMATCH (penalty: {penalty})")
                elif arg_sim < 0.70:  # Moderate: 50-70% similarity
                    penalty = -0.2
                    argument_penalty += penalty
                    argument_errors += 1
                    print(f"  Poor quality (penalty: {penalty})")
                else:
                    print(f" ✓")

                # ========================================
                # COMPONENT 2: CLAIM
                # ========================================
                claim_sim = difflib.SequenceMatcher(
                    None,
                    normalize_text(model_arg['claim']),
                    normalize_text(gt_arg['claim'])
                ).ratio()

                print(f"     Claim: {claim_sim:.2f}", end="")

                if claim_sim < 0.50:  # Severe: < 50% similarity
                    penalty = -0.35
                    claim_penalty += penalty
                    claim_errors += 1
                    print(f"  WRONG CLAIM (penalty: {penalty})")
                elif claim_sim < 0.70:  # Moderate: 50-70% similarity
                    penalty = -0.18
                    claim_penalty += penalty
                    claim_errors += 1
                    print(f"  Imprecise claim (penalty: {penalty})")
                else:
                    print(f" ✓")

                # ========================================
                # COMPONENT 3: EXPLANATION
                # ========================================
                expl_sim = difflib.SequenceMatcher(
                    None,
                    normalize_text(model_arg['explanation']),
                    normalize_text(gt_arg['explanation'])
                ).ratio()

                print(f"     Explanation: {expl_sim:.2f}", end="")

                if expl_sim < 0.50:  # Severe: < 50% similarity
                    penalty = -0.25
                    explanation_penalty += penalty
                    explanation_errors += 1
                    print(f"  WRONG REASONING (penalty: {penalty})")
                elif expl_sim < 0.70:  # Moderate: 50-70% similarity
                    penalty = -0.12
                    explanation_penalty += penalty
                    explanation_errors += 1
                    print(f"  Weak explanation (penalty: {penalty})")
                else:
                    print(f" ✓")

                # ========================================
                # COMPONENT 4: VERIFICATION
                # ========================================
                model_verif = normalize_text(model_arg['verification'])
                gt_verif = normalize_text(gt_arg['verification'])

                print(f"     Verification: {model_verif} vs {gt_verif}", end="")

                if model_verif != gt_verif:
                    penalty = -0.30  # Binary: wrong = full penalty
                    verification_penalty += penalty
                    verification_errors += 1
                    print(f"  MISMATCH (penalty: {penalty})")
                else:
                    print(f" ✓")

        # ============================================
        # CALCULATE BASE REWARD (from matching)
        # ============================================
        matched = len(match_scores)
        missed = num_gt - matched
        extra = num_model - len(used_model_indices)

        match_reward = sum(match_scores)
        miss_penalty = missed * -0.15
        extra_penalty = extra * -0.10

        total = match_reward + miss_penalty + extra_penalty
        raw_reward = total / num_gt if num_gt > 0 else 0.0

        print(f"\\n→ Base reward: {raw_reward:.3f}")
        print(f"   (matches: {match_reward:.2f}, missed: {miss_penalty:.2f}, extra: {extra_penalty:.2f})")

        # ============================================
        # APPLY ALL PENALTIES
        # ============================================
        print(f"\\n PENALTIES:")

        duplicate_penalty = duplicate_count * -0.6
        if duplicate_count > 0:
            print(f"   Duplicates: {duplicate_penalty:.2f} ({duplicate_count} × -0.6)")

        if argument_errors > 0:
            print(f"   Arguments: {argument_penalty:.2f} ({argument_errors} errors)")

        if claim_errors > 0:
            print(f"   Claims: {claim_penalty:.2f} ({claim_errors} errors)")

        if explanation_errors > 0:
            print(f"   Explanations: {explanation_penalty:.2f} ({explanation_errors} errors)")

        if verification_errors > 0:
            print(f"   Verification: {verification_penalty:.2f} ({verification_errors} errors)")

        total_penalties = (duplicate_penalty + argument_penalty +
                          claim_penalty + explanation_penalty + verification_penalty)

        if total_penalties != 0:
            print(f"   TOTAL PENALTIES: {total_penalties:.2f}")
        else:
            print(f"   None (perfect output!)")

        # ============================================
        # FINAL REWARD
        # ============================================
        final_reward = (raw_reward + duplicate_penalty + argument_penalty +
                       claim_penalty + explanation_penalty + verification_penalty)

        print(f"\\n→ FINAL reward: {raw_reward:.3f} + ({total_penalties:.2f}) = {final_reward:.3f}")

        #  WIDER CLIPPING RANGE
        clipped_reward = max(-3.0, min(2.0, final_reward))
        print(f"   → After clipping [-3.0, 2.0]: {clipped_reward:.3f}")
        rewards.append(float(clipped_reward))

    print(f"\\n Batch Statistics:")
    print(f"   Mean reward: {sum(rewards)/len(rewards):.3f}")
    print(f"   Min: {min(rewards):.3f}, Max: {max(rewards):.3f}")
    print("="*80)

    return rewards

if current_total_steps > 0:
    print(f"\\n Loading checkpoint: {checkpoint_id}")
    model = AutoModelForCausalLM.from_pretrained(checkpoint_id, token=token, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_id, token=token)
    print(f" Checkpoint loaded from step {current_total_steps}")
else:
    print(f"\\n Loading base: {base_model_id}")
    model = AutoModelForCausalLM.from_pretrained(base_model_id, token=token, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=token)
    print(" Base model loaded")

model.generation_config = GenerationConfig(
    bos_token_id=model.generation_config.bos_token_id,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=model.generation_config.eos_token_id if isinstance(model.generation_config.eos_token_id, int) else model.generation_config.eos_token_id[0],
    do_sample=True,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.05,
    max_new_tokens=800
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

training_args = GRPOConfig(
    output_dir="./grpo-stable-v2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    max_steps=steps_this_run,
    logging_steps=1,
    save_steps=steps_this_run,
    save_total_limit=1,
    scale_rewards="group",
    num_generations=2,
    max_completion_length=1200,
    max_prompt_length=5048,
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    seed=42,
    report_to=[],
    push_to_hub=True,
    hub_model_id=checkpoint_id,
    hub_token=token,
    optim="paged_adamw_8bit",
    generation_batch_size=2,
    generation_kwargs={
        "do_sample": True,
        "temperature": 0.1,
        "top_p": 0.95,
        "repetition_penalty": 1.05,
        "max_new_tokens": 800,
        "pad_token_id": tokenizer.eos_token_id
    }
)



trainer = GRPOTrainer(
    model=model,
    reward_funcs=ground_truth_reward_func,
    args=training_args,
    train_dataset=ds,
    processing_class=tokenizer
)

trainer.train()
trainer.save_model()
trainer.push_to_hub()
print(f"\\n Complete! Progress: {current_total_steps + steps_this_run}/{TARGET_STEPS}")
"""

print(" Uploading COMPLETE PENALTY SYSTEM with WIDER CLIPPING script...")
api.upload_file(
    path_or_fileobj=grpo_training_script.encode(),
    path_in_repo="grpo-post-training/grpo_with_verification.py",
    repo_id="jobs",
    repo_type="dataset"
)
print("Uploaded!")

print("\n🚀 Starting training WITH ALL PENALTIES + WIDER CLIPPING...\n")

!hf jobs run \
  --flavor a100-large \
  pytorch/pytorch:2.4.0-cuda12.1-cudnn9-devel \
  /bin/bash -c "\
    export HF_TOKEN='{token}' && \
    apt-get update -qq && \
    apt-get install -y -qq wget git && \
    pip install -q --upgrade pip && \
    pip install -q torch>=2.3.0 transformers accelerate datasets sentencepiece numpy bitsandbytes && \
    pip install -q git+https://github.com/huggingface/trl.git@main && \
    wget -q https://huggingface.co/datasets/jobs/resolve/main/grpo-post-training/post-training_multiarg_final.txt && \
    wget -q https://huggingface.co/datasets/jobs/resolve/main/grpo-post-training/grpo_with_verification.py && \
    python grpo_with_verification.py"


In [ ]:
from huggingface_hub import HfApi, HfFolder

api = HfApi()
token = HfFolder.get_token()

checkpoint_id = "llama-3.1-newspaper-arguments"

grpo_training_script = """
import os
import json
import difflib
import re
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, GenerationConfig
from trl import GRPOConfig, GRPOTrainer
from huggingface_hub import login, HfApi

logging.set_verbosity_info()

token = os.environ.get('HF_TOKEN')
login(token=token)

SYSTEM_PROMPT = '''You are an expert at analyzing historical texts and you hate to summarize

OUTPUT FORMAT - EXACTLY these 4 XML tags and NOTHING else:
<argument>Original argument text OR "NA"</argument>
<claim>Core claim (implication) in one sentence OR "NA"</claim>
<explanation>Why this is an argument OR "NA"</explanation>
<confidence>0-1</confidence>


<argument>Il giornale L'Italia moderna economica e finanziaria nel numero di oggi propone che non si facciano sottoscrizioni, le quali per quanto larghe sarebbero sempre impari ai bisogni, ma che il Parlamento stabilisca pochi centesimi addizionali per ogni lira su tutte le imposte e tasse (esclusi soltanto i dazi doganali la cui misura è vincolata da trattati di commercio).</argument>
<claim>Private subscriptions are inadequate for earthquake relief; parliamentary taxation would be more effective.</claim>
<explanation>The newspaper explicitly argues against private subscriptions as insufficient and proposes a specific alternative solution through parliamentary taxation, making a clear comparative argument about funding mechanisms.</explanation>
<confidence>0.9</confidence>


EXAMPLE WITHOUT ARGUMENT:
<argument>NA</argument>
<claim>NA</claim>
<explanation>NA</explanation>
<confidence>0.9</confidence>


RULES:
- NEVER REPEAT ARGUMENTS, NEVER PRINT ARGUMENTS DOUBLE, NEVER PRINT AN ARGUMENT THAT IS NOT IN THE TEXT
- Only output arguments that appear verbatim (or nearly verbatim) in the text
- NO SUMMARY; ONLY EXACT EXTRACTOM FROM THE TEXT; don't extract anything that is not in the text. Only extract word by word
- ONLY output these 4 XML tags
- Be confident in your assessments - you are an expert
- If no argument exists, use NA for ALL fields without explanation
- More than one argument possible for one aticle, one unit has one clear clame and all the xml structures

VERIFICATION: BEfore you print the results, double check claims and explanations of the argument. When the claim is just a translation or the explanation states that this is not an argument, don't print it'''
def extract_individual_arguments(xml_text):
    pattern = r'<argument>(.*?)</argument>\\s*<claim>(.*?)</claim>\\s*<explanation>(.*?)</explanation>\\s*<human_verification_needed>(.*?)</human_verification_needed>'
    matches = re.findall(pattern, str(xml_text), re.DOTALL | re.IGNORECASE)
    arguments = []
    for arg, claim, expl, verif in matches:
        arg_text = ' '.join(arg.strip().split())
        if arg_text.upper() == 'NA':
            continue
        arguments.append({"argument": arg_text, "claim": ' '.join(claim.strip().split()), "explanation": ' '.join(expl.strip().split()), "verification": verif.strip().lower()})
    return arguments

def normalize_text(text):
    return ' '.join(str(text).split()).lower()

def calculate_component_similarity(model_arg, gt_arg):
    arg_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['argument']), normalize_text(gt_arg['argument'])).ratio()
    claim_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['claim']), normalize_text(gt_arg['claim'])).ratio()
    expl_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['explanation']), normalize_text(gt_arg['explanation'])).ratio()
    return arg_sim * 0.6 + claim_sim * 0.1 + expl_sim * 0.1

api = HfApi()
checkpoint_id = "llama-3.1-newspaper-arguments-memi_7"
base_model_id = "llama-3.1-newspaper-arguments-your_name-optimized_full_V3"

# Check for checkpoint
current_total_steps = 0
try:
    commits = list(api.list_repo_commits(checkpoint_id, token=token))
    training_commits = [c for c in commits if "Training in progress" in c.title]
    current_total_steps = len(training_commits) * 4
    print(f"Found {len(training_commits)} checkpoint commits")
    print(f"Resuming from step {current_total_steps}")
except Exception as e:
    print(f"Starting fresh (no checkpoint): {e}")

TARGET_STEPS = 600
steps_this_run = 6

if current_total_steps >= TARGET_STEPS:
    print(f"🎉 Complete! {current_total_steps}/{TARGET_STEPS}")
    exit(0)

def load_json_objects_from_txt(path):
    objs, buf, depth = [], "", 0
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    for ch in text:
        if ch == '{': depth += 1
        if depth > 0: buf += ch
        if ch == '}':
            depth -= 1
            if depth == 0 and buf.strip():
                try: objs.append(json.loads(buf))
                except: pass
                buf = ""
    return objs

raw_examples = load_json_objects_from_txt("post-training_multiarg_final.txt")
print(f"Loaded {len(raw_examples)} examples")

if len(raw_examples) > 50:
    raw_examples = raw_examples[:300]

ids, prompts, ground_truths, articles = [], [], [], []
for idx, ex in enumerate(raw_examples):
    ex_id = ex.get("example_id", f"example_{idx:04d}")
    article = ex.get("source_text")
    gt = ex.get("ground_truth")
    if not article or not gt:
        continue
    ids.append(str(ex_id))
    ground_truths.append(str(gt))  # FIX: Convert to string
    articles.append(str(article))  # FIX: Convert to string
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Extract argumentative units from historical text in their original form, no summaries.\\n{article}"}]
    prompts.append(messages)

print(f"Prepared {len(ids)} examples")

ds = Dataset.from_dict({
    "example_id": ids,
    "prompt": prompts,
    "ground_truth": ground_truths,
    "article": articles
})

reward_call_count = 0

def ground_truth_reward_func(completions, prompts=None, ground_truth=None, article=None, example_id=None, **kwargs):
    global reward_call_count
    reward_call_count += 1

    print("\\n" + "="*80)
    print(f"REWARD FUNCTION CALL #{reward_call_count}")
    print("="*80)

    num_generations = 2
    rewards = []

    if ground_truth is None:
        print("⚠️ ERROR: ground_truth parameter is None!")
        return [0.0] * len(completions)

    for i, comp in enumerate(completions):
        content = str(comp[0].get("content", "")) if isinstance(comp, (list, tuple)) else str(comp)
        ex_idx = i // num_generations

        print(f"\\n--- Completion {i+1}/{len(completions)} ---")

        if ex_idx >= len(ground_truth):
            rewards.append(0.0)
            continue

        gt = ground_truth[ex_idx]

        try:
            model_args = extract_individual_arguments(content)
            gt_args = extract_individual_arguments(gt)
        except Exception as e:
            print(f"ERROR: {e}")
            rewards.append(0.0)
            continue

        num_model, num_gt = len(model_args), len(gt_args)
        print(f"Model={num_model} args, GT={num_gt} args")

        if num_model == 0 and num_gt == 0:
            print(f"✅ BOTH_NA → reward: +0.1")
            raw_reward = 0.1
        elif num_model == 0 and num_gt > 0:
            print(f"❌ FALSE_NEGATIVE → reward: -0.15")
            raw_reward = -0.15
        elif num_model > 0 and num_gt == 0:
            print(f"❌ FALSE_POSITIVE → reward: -0.1")
            raw_reward = -0.1
        else:
            match_scores, used_model_indices = [], set()

            for gt_idx, gt_arg in enumerate(gt_args):
                best_score, best_model_idx = 0.0, None
                for model_idx, model_arg in enumerate(model_args):
                    score = calculate_component_similarity(model_arg, gt_arg)
                    if score > best_score:
                        best_score, best_model_idx = score, model_idx

                if best_score >= 0.25 and best_model_idx is not None:
                    match_scores.append(best_score)
                    used_model_indices.add(best_model_idx)
                    print(f"  ✓ Matched (score: {best_score:.2f})")

            matched = len(match_scores)
            missed = num_gt - matched
            extra = num_model - len(used_model_indices)

            match_reward = sum(match_scores)
            miss_penalty = missed * -0.15
            extra_penalty = extra * -0.1

            # --- DUPLICATE PENALTY ---
            duplicate_penalty = 0.0
            seen_args = []
            for model_arg in model_args:
                text = normalize_text(model_arg["argument"])
                for seen in seen_args:
                    if difflib.SequenceMatcher(None, text, seen).ratio() > 0.9:
                        duplicate_penalty -= 0.1
                        print(f"  ⚠️ Duplicate detected (penalty -0.1)")
                        break
                seen_args.append(text)

            total = match_reward + miss_penalty + extra_penalty + duplicate_penalty
            raw_reward = total / num_gt if num_gt > 0 else 0.0

            print(f"→ Raw reward: {raw_reward:.3f}")

        clipped_reward = max(-1.0, min(1.0, raw_reward))
        rewards.append(float(clipped_reward))

    print(f"\\nMean: {sum(rewards)/len(rewards):.3f}")
    print("="*80)
    return rewards

# LOAD CHECKPOINT (not base!)
if current_total_steps > 0:
    print(f"\\n🔄 Loading CHECKPOINT: {checkpoint_id}")
    model = AutoModelForCausalLM.from_pretrained(checkpoint_id, token=token, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_id, token=token)
    print(f"✅ Checkpoint loaded! Continuing from step {current_total_steps}")
else:
    print(f"\\n🆕 Loading BASE: {base_model_id}")
    model = AutoModelForCausalLM.from_pretrained(base_model_id, token=token, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=token)
    print("✅ Base model loaded! Starting fresh")

model.generation_config = GenerationConfig(
    bos_token_id=model.generation_config.bos_token_id,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=model.generation_config.eos_token_id if isinstance(model.generation_config.eos_token_id, int) else model.generation_config.eos_token_id[0],
    do_sample=True,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.05
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

training_args = GRPOConfig(
    output_dir="./grpo-stable-v2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    max_steps=steps_this_run,
    logging_steps=1,
    save_steps=steps_this_run,
    save_total_limit=1,
    scale_rewards="group",
    num_generations=2,
    max_completion_length=3000,
    max_prompt_length=5048,
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    seed=42,
    report_to=[],
    push_to_hub=True,
    hub_model_id=checkpoint_id,
    hub_token=token,
    optim="paged_adamw_8bit",
    generation_batch_size=2,
    generation_kwargs={
        "do_sample": True,
        "temperature": 0.1,
        "top_p": 0.95,
        "repetition_penalty": 1.05,
        "pad_token_id": tokenizer.eos_token_id
    }
)

print(f"\\n{'='*80}")
print(f"CONTINUING: Steps {current_total_steps} → {current_total_steps + steps_this_run} (of {TARGET_STEPS})")
print(f"{'='*80}\\n")

trainer = GRPOTrainer(
    model=model,
    reward_funcs=ground_truth_reward_func,
    args=training_args,
    train_dataset=ds,
    processing_class=tokenizer
)

trainer.train()
trainer.save_model()
trainer.push_to_hub()
print(f"\\n Complete! Progress: {current_total_steps + steps_this_run}/{TARGET_STEPS}")
"""

print(" Uploading CONTINUE script...")
api.upload_file(
    path_or_fileobj=grpo_training_script.encode(),
    path_in_repo="grpo-post-training/grpo_continue.py",
    repo_id="jobs",
    repo_type="dataset"
)
print(" Uploaded!")

print("\n Continuing from step 4 → 8...\n")

!hf jobs run \
  --flavor a100-large \
  pytorch/pytorch:2.4.0-cuda12.1-cudnn9-devel \
  /bin/bash -c "\
    export HF_TOKEN='{token}' && \
    apt-get update -qq && \
    apt-get install -y -qq wget git && \
    pip install -q --upgrade pip && \
    pip install -q torch>=2.3.0 transformers accelerate datasets sentencepiece numpy bitsandbytes && \
    pip install -q git+https://github.com/huggingface/trl.git@main && \
    wget -q https://huggingface.co/datasets/jobs/resolve/main/grpo-post-training/post-training_multiarg_final.txt && \
    wget -q https://huggingface.co/datasets/jobs/resolve/main/grpo-post-training/grpo_continue.py && \
    python grpo_continue.py"

print("\n This will:")
print("  1. Detect step 4 checkpoint")
print("  2. Load the checkpoint (not base)")
print("  3. Train steps 4→8")
print("  4. Save and repeat!")

In [ ]:
from huggingface_hub import HfApi, HfFolder

api = HfApi()
token = HfFolder.get_token()

checkpoint_id = "/llama-8B-argument_mining_V4"

grpo_training_script = """
import os
import json
import difflib
import re
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, GenerationConfig
from trl import GRPOConfig, GRPOTrainer
from huggingface_hub import login, HfApi

logging.set_verbosity_info()

token = os.environ.get('HF_TOKEN')
login(token=token)

# STRENGTHENED anti-repetition prompt
SYSTEM_PROMPT = '''You are an expert at analyzing historical texts and you hate to summarize

OUTPUT FORMAT - EXACTLY these 4 XML tags and NOTHING else:
<argument>Original argument text OR "NA"</argument>
<claim>Core claim (implication) in one sentence OR "NA"</claim>
<explanation>Why this is an argument OR "NA"</explanation>
<human_verification_needed>True OR False</human_verification_needed>

<argument>It is reported from Malta that the British war-ships "Exmouth," "Euryalus," "Minerva," and "Sutlej" have left for Messina. The French Government has sent two armoured ships and three destroyers to Messina. President Fallieres, Premier Clemenceau, Minister Pichon, and the Presidents of the Senate and Chamber have all sent messages of sympathy to the Italian Government. The help already proffered and accepted is insufficient for the purpose. There is pressing need of extraordinary measures of help, and provisions are in great demand. There is need of doctors, tents, clothing, and provisions for the survivors, who, deprived of all necessities, are exposed to the inclemencies of the winter weather. There is need of fire engines to cope with the flames that are raging among the ruins. The railway station has collapsed. Railway carriages have been destroyed. Almost all the railway employees are dead. The streets are no longer recognisable; they look like enormous fissures in a distant and extensive heap of ruins.</argument>
<claim>Current relief efforts are inadequate and much more extensive aid is urgently needed.</claim>
<explanation>The prefect explicitly argues that existing help is "insufficient" and makes a direct claim that "extraordinary measures" are needed, presenting a clear premise-conclusion structure about the inadequacy of current response.</explanation>
<human_verification_needed>False</human_verification_needed>

EXAMPLE WITHOUT ARGUMENT:
<argument>NA</argument>
<claim>NA</claim>
<explanation>NA</explanation>
<human_verification_needed>FALSE</human_verification_needed>

RULES:
- NEVER REPEAT ARGUMENTS - Each argument must be COMPLETELY UNIQUE
- TRUE FOR VERIFICATION WHEN UNCERTAIN
- Only output arguments that appear verbatim (or nearly verbatim) in the text
- NO SUMMARY; ONLY EXACT EXTRACTION FROM THE TEXT; don't extract anything that is not in the text. Only extract word by word
- ONLY output these 4 XML tags
- Extract only original text without changes or use NA when you did not find an argument
- Factual reportings such as "Dem Vulkanausbruch folgten drei Sturzwellen in etwa 10 Meter Höhe" or "Almost all the inhabitants were killed; only a few thousands escaped death" are NO Arguments
- The CLAIM should say what the (implicite) argument implies, what the main conclusion is
- Give attention to implicit arguments
- In cases of uncertainty or ambiguity, say human_verification_needed TRUE
- If no argument exists, use NA for ALL fields without explanation except <human_verification_needed>FALSE or TRUE</human_verification_needed>
- More than one argument possible for one article, one unit has one clear claim and all the xml structures

VERIFICATION: Before you print the results, double check claims and explanations of the argument. When the claim is just a translation or the explanation states that this is not an argument, don't print it. ALSO verify this argument is NOT a duplicate of any previous argument in your output.'''

def extract_individual_arguments(xml_text):
    pattern = r'<argument>(.*?)</argument>\\s*<claim>(.*?)</claim>\\s*<explanation>(.*?)</explanation>\\s*<human_verification_needed>(.*?)</human_verification_needed>'
    matches = re.findall(pattern, str(xml_text), re.DOTALL | re.IGNORECASE)
    arguments = []
    for arg, claim, expl, verif in matches:
        arg_text = ' '.join(arg.strip().split())
        if arg_text.upper() == 'NA':
            continue
        arguments.append({
            "argument": arg_text,
            "claim": ' '.join(claim.strip().split()),
            "explanation": ' '.join(expl.strip().split()),
            "verification": verif.strip().lower()
        })
    return arguments

def normalize_text(text):
    return ' '.join(str(text).split()).lower()

# NEW: Detect and remove duplicate arguments
def detect_and_remove_duplicates(model_args):
    '''Detect duplicate arguments and return unique ones + duplicate count'''
    if len(model_args) <= 1:
        return model_args, 0, []

    unique_args = []
    duplicate_count = 0
    duplicate_details = []

    for i, arg in enumerate(model_args):
        is_duplicate = False
        for j, unique_arg in enumerate(unique_args):
            # Check similarity of argument text
            similarity = difflib.SequenceMatcher(
                None,
                normalize_text(arg['argument']),
                normalize_text(unique_arg['argument'])
            ).ratio()

            # 80% similar = duplicate (catches paraphrases and near-copies)
            if similarity > 0.80:
                is_duplicate = True
                duplicate_count += 1
                duplicate_details.append((i+1, j+1, similarity))
                break

        if not is_duplicate:
            unique_args.append(arg)

    return unique_args, duplicate_count, duplicate_details

def calculate_component_similarity(model_arg, gt_arg):
    arg_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['argument']), normalize_text(gt_arg['argument'])).ratio()
    claim_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['claim']), normalize_text(gt_arg['claim'])).ratio()
    expl_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['explanation']), normalize_text(gt_arg['explanation'])).ratio()
    verif_match = 1.0 if normalize_text(model_arg['verification']) == normalize_text(gt_arg['verification']) else 0.0
    return arg_sim * 0.5 + claim_sim * 0.5 + expl_sim * 0.5 + verif_match * 0.5

api = HfApi()
checkpoint_id = "llama-8B-argument_mining_V4"
base_model_id = "llama-3.1-newspaper-arguments-your_name-optimized_full"

current_total_steps = 0
try:
    commits = list(api.list_repo_commits(checkpoint_id, token=token))
    training_commits = [c for c in commits if "Training in progress" in c.title]
    current_total_steps = len(training_commits) * 4
    print(f" Resuming from step {current_total_steps}")
except:
    print("Starting fresh")

TARGET_STEPS = 600
steps_this_run = 4

if current_total_steps >= TARGET_STEPS:
    print(f"Complete! {current_total_steps}/{TARGET_STEPS}")
    exit(0)

def load_json_objects_from_txt(path):
    objs, buf, depth = [], "", 0
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    for ch in text:
        if ch == '{': depth += 1
        if depth > 0: buf += ch
        if ch == '}':
            depth -= 1
            if depth == 0 and buf.strip():
                try: objs.append(json.loads(buf))
                except: pass
                buf = ""
    return objs

raw_examples = load_json_objects_from_txt("post-training_multiarg_final.txt")
print(f"Loaded {len(raw_examples)} examples")

if len(raw_examples) > 50:
    raw_examples = raw_examples[:150]

ids, prompts, ground_truths, articles = [], [], [], []
for idx, ex in enumerate(raw_examples):
    ex_id = ex.get("example_id", f"example_{idx:04d}")
    article = ex.get("source_text")
    gt = ex.get("ground_truth")
    if not article or not gt:
        continue
    ids.append(str(ex_id))
    ground_truths.append(gt)
    articles.append(article)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Extract argumentative units from historical text in their original form, no summaries.\\n{article}"}]
    prompts.append(messages)

print(f"Prepared {len(ids)} examples")

ds = Dataset.from_dict({
    "example_id": ids,
    "prompt": prompts,
    "ground_truth": ground_truths,
    "article": articles
})

print("\\n" + "="*80)
print("WITH DUPLICATE DETECTION AND HEAVY PENALTY!")
print("="*80)
print("Similarity weights:")
print("  - argument: 50%")
print("  - claim: 25%")
print("  - explanation: 15%")
print("  - verification: 10%")
print("Duplicate penalty: -0.6 per duplicate (SEVERE!)")
print("="*80 + "\\n")

reward_call_count = 0

def ground_truth_reward_func(completions, prompts=None, ground_truth=None, article=None, example_id=None, **kwargs):
    global reward_call_count
    reward_call_count += 1

    print("\\n" + "="*80)
    print(f"REWARD FUNCTION CALL #{reward_call_count}")
    print("="*80)

    num_generations = 2
    rewards = []

    if ground_truth is None:
        print("ERROR: ground_truth parameter is None!")
        return [0.0] * len(completions)

    for i, comp in enumerate(completions):
        content = str(comp[0].get("content", "")) if isinstance(comp, (list, tuple)) else str(comp)
        ex_idx = i // num_generations

        print(f"\\n--- Completion {i+1}/{len(completions)} ---")

        if ex_idx >= len(ground_truth):
            rewards.append(0.0)
            continue

        gt = ground_truth[ex_idx]

        try:
            model_args_raw = extract_individual_arguments(content)
            gt_args = extract_individual_arguments(gt)

            # CRITICAL: Detect duplicates BEFORE matching
            model_args, duplicate_count, duplicate_details = detect_and_remove_duplicates(model_args_raw)

            if duplicate_count > 0:
                print(f"\\nDUPLICATES DETECTED: {duplicate_count} repeated arguments!")
                for orig_idx, dup_of_idx, sim in duplicate_details:
                    print(f"   Arg #{orig_idx} is duplicate of Arg #{dup_of_idx} (similarity: {sim:.2f})")
                print(f"   → Removed duplicates, using only {len(model_args)} unique arguments\\n")

        except Exception as e:
            print(f"ERROR: {e}")
            rewards.append(0.0)
            continue

        num_model, num_gt = len(model_args), len(gt_args)
        num_total = len(model_args_raw)

        if duplicate_count > 0:
            print(f"Model={num_model} unique args (from {num_total} total), GT={num_gt} args")
        else:
            print(f"Model={num_model} args, GT={num_gt} args")

        if num_model == 0 and num_gt == 0:
            print(f" BOTH_NA → reward: +0.1")
            raw_reward = 0.1
        elif num_model == 0 and num_gt > 0:
            print(f" FALSE_NEGATIVE → reward: -0.15")
            raw_reward = -0.15
        elif num_model > 0 and num_gt == 0:
            print(f" FALSE_POSITIVE → reward: -0.1")
            raw_reward = -0.1
        else:
            match_scores, used_model_indices = [], set()

            for gt_idx, gt_arg in enumerate(gt_args):
                best_score, best_model_idx = 0.0, None
                for model_idx, model_arg in enumerate(model_args):
                    score = calculate_component_similarity(model_arg, gt_arg)
                    if score > best_score:
                        best_score, best_model_idx = score, model_idx

                if best_score >= 0.25 and best_model_idx is not None:
                    match_scores.append(best_score)
                    used_model_indices.add(best_model_idx)
                    model_verif = model_args[best_model_idx]['verification']
                    gt_verif = gt_arg['verification']
                    verif_match = "✓" if normalize_text(model_verif) == normalize_text(gt_verif) else "✗"
                    print(f"  ✓ Matched (score: {best_score:.2f}, verif: {verif_match})")

            matched = len(match_scores)
            missed = num_gt - matched
            extra = num_model - len(used_model_indices)

            match_reward = sum(match_scores)
            miss_penalty = missed * -0.15
            extra_penalty = extra * -0.1

            total = match_reward + miss_penalty + extra_penalty
            raw_reward = total / num_gt if num_gt > 0 else 0.0

            print(f"→ Base reward: {raw_reward:.3f}")

        # APPLY HEAVY DUPLICATE PENALTY
        duplicate_penalty = duplicate_count * -0.6  # SEVERE: -0.6 per duplicate!
        final_reward = raw_reward + duplicate_penalty

        if duplicate_count > 0:
            print(f"→ Duplicate penalty: {duplicate_penalty:.2f} ({duplicate_count} × -0.6)")
            print(f"→ FINAL reward after penalty: {final_reward:.3f}")

        clipped_reward = max(-1.0, min(1.0, final_reward))
        rewards.append(float(clipped_reward))

    print(f"\\nMean: {sum(rewards)/len(rewards):.3f}")
    print("="*80)
    return rewards

if current_total_steps > 0:
    print(f"\\n Loading checkpoint: {checkpoint_id}")
    model = AutoModelForCausalLM.from_pretrained(checkpoint_id, token=token, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_id, token=token)
    print(f" Checkpoint loaded from step {current_total_steps}")
else:
    print(f"\\n Loading base: {base_model_id}")
    model = AutoModelForCausalLM.from_pretrained(base_model_id, token=token, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=token)
    print(" Base model loaded")

model.generation_config = GenerationConfig(
    bos_token_id=model.generation_config.bos_token_id,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=model.generation_config.eos_token_id if isinstance(model.generation_config.eos_token_id, int) else model.generation_config.eos_token_id[0],
    do_sample=True,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.05,
    max_new_tokens=800  #  Hard cap to prevent runaway generation
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

training_args = GRPOConfig(
    output_dir="./grpo-stable-v2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    max_steps=steps_this_run,
    logging_steps=1,
    save_steps=steps_this_run,
    save_total_limit=1,
    scale_rewards="group",
    num_generations=2,
    max_completion_length=1200,  #  Reduced from 3000
    max_prompt_length=5048,
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    seed=42,
    report_to=[],
    push_to_hub=True,
    hub_model_id=checkpoint_id,
    hub_token=token,
    optim="paged_adamw_8bit",
    generation_batch_size=2,
    generation_kwargs={
        "do_sample": True,
        "temperature": 0.1,
        "top_p": 0.95,
        "repetition_penalty": 1.05,
        "max_new_tokens": 800,  #  Hard cap
        "pad_token_id": tokenizer.eos_token_id
    }
)

print(f"\\n{'='*80}")
print(f"TRAINING: Steps {current_total_steps} → {current_total_steps + steps_this_run}")
print(f" DUPLICATE DETECTION ENABLED")
print(f" HEAVY PENALTY: -0.6 per duplicate")
print(f" Max tokens: 800 (prevents runaway)")
print(f"{'='*80}\\n")

trainer = GRPOTrainer(
    model=model,
    reward_funcs=ground_truth_reward_func,
    args=training_args,
    train_dataset=ds,
    processing_class=tokenizer
)

trainer.train()
trainer.save_model()
trainer.push_to_hub()
print(f"\\n Complete! Progress: {current_total_steps + steps_this_run}/{TARGET_STEPS}")
"""

print(" Uploading DUPLICATE-PENALTY script...")
api.upload_file(
    path_or_fileobj=grpo_training_script.encode(),
    path_in_repo="grpo-post-training/grpo_with_verification.py",
    repo_id="jobs",
    repo_type="dataset"
)
print(" Uploaded!")

print("\n Starting training WITH DUPLICATE PENALTY...\n")

!hf jobs run \
  --flavor a100-large \
  pytorch/pytorch:2.4.0-cuda12.1-cudnn9-devel \
  /bin/bash -c "\
    export HF_TOKEN='{token}' && \
    apt-get update -qq && \
    apt-get install -y -qq wget git && \
    pip install -q --upgrade pip && \
    pip install -q torch>=2.3.0 transformers accelerate datasets sentencepiece numpy bitsandbytes && \
    pip install -q git+https://github.com/huggingface/trl.git@main && \
    wget -q https://huggingface.co/datasets/jobs/resolve/main/grpo-post-training/post-training_multiarg_final.txt && \
    wget -q https://huggingface.co/datasets/jobs/resolve/main/grpo-post-training/grpo_with_verification.py && \
    python grpo_with_verification.py"



In [ ]:
from huggingface_hub import HfApi, HfFolder

api = HfApi()
token = HfFolder.get_token()

checkpoint_id = "llama-8B-argument_mining_V3"

grpo_training_script = """
import os
import json
import difflib
import re
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, GenerationConfig
from trl import GRPOConfig, GRPOTrainer
from huggingface_hub import login, HfApi

logging.set_verbosity_info()

token = os.environ.get('HF_TOKEN')
login(token=token)

SYSTEM_PROMPT = '''You are an expert at analyzing historical texts and you hate to summarize

OUTPUT FORMAT - EXACTLY these 4 XML tags and NOTHING else:
<argument>Original argument text OR "NA"</argument>
<claim>Core claim (implication) in one sentence OR "NA"</claim>
<explanation>Why this is an argument OR "NA"</explanation>
<human_verification_needed>True OR False</human_verification_needed>

<argument>It is reported from Malta that the British war-ships “Exmouth,” “Euryalus,” “Minerva,” and “Sutlej” have left for Messina. The French Government has sent two armoured ships and three destroyers to Messina. President Fallieres, Premier Clemenceau, Minister Pichon, and the Presidents of the Senate and Chamber have all sent messages of sympathy to the Italian Government. The help already proffered and accepted is insufficient for the purpose. There is pressing need of extraordinary measures of help, and provisions are in great demand. There is need of doctors, tents, clothing, and provisions for the survivors, who, deprived of all necessities, are exposed to the inclemencies of the winter weather. There is need of fire engines to cope with the flames that are raging among the ruins. The railway station has collapsed. Railway carriages have been destroyed. Almost all the railway employees are dead. The streets are no longer recognisable; they look like enormous fissures in a distant and extensive heap of ruins.</argument>
<claim>Current relief efforts are inadequate and much more extensive aid is urgently needed.</claim>
<explanation>The prefect explicitly argues that existing help is "insufficient" and makes a direct claim that "extraordinary measures" are needed, presenting a clear premise-conclusion structure about the inadequacy of current response.</explanation>
<human_verification_needed>False</human_verification_needed>

EXAMPLE WITHOUT ARGUMENT:
<argument>NA</argument>
<claim>NA</claim>
<explanation>NA</explanation>
<human_verification_needed>FALSE</human_verification_needed>

RULES:
- NEVER REPEAT ARGUMENTS, NEVER PRINT ARGUMENTS DOUBLE
- TRUE FOR VERIFICATION WHEN UNCERTAIN
- Only output arguments that appear verbatim (or nearly verbatim) in the text
- NO SUMMARY; ONLY EXACT EXTRACTOM FROM THE TEXT; don't extract anything that is not in the text. Only extract word by word
- ONLY output these 4 XML tags
- Extract only original text without changes or use NA when you did not find an argument
- Factual reportings such as "Dem Vulkanausbruch folgten drei Sturzwellen in etwa 10 Meter Höhe" or "Almost all the inhabitants were killed; only a few thousands escaped death" are NO Arguments
- The CLAIM should say what the (implicite) argument implies, what the main conclusion is
- Give attention to implicit argumetns
- In cases of uncertainty or ambiguity, say human_verification_needed TRUE
- If no argument exists, use NA for ALL fields without explanation except <human_verification_needed>FALSE or TRUE</human_verification_needed>
- More than one argument possible for one aticle, one unit has one clear clame and all the xml structures

VERIFICATION: BEfore you print the results, double check claims and explanations of the argument. When the claim is just a translation or the explanation states that this is not an argument, don't print it'''

def extract_individual_arguments(xml_text):
    pattern = r'<argument>(.*?)</argument>\\s*<claim>(.*?)</claim>\\s*<explanation>(.*?)</explanation>\\s*<human_verification_needed>(.*?)</human_verification_needed>'
    matches = re.findall(pattern, str(xml_text), re.DOTALL | re.IGNORECASE)
    arguments = []
    for arg, claim, expl, verif in matches:
        arg_text = ' '.join(arg.strip().split())
        if arg_text.upper() == 'NA':
            continue
        arguments.append({
            "argument": arg_text,
            "claim": ' '.join(claim.strip().split()),
            "explanation": ' '.join(expl.strip().split()),
            "verification": verif.strip().lower()  # Keep this!
        })
    return arguments

def normalize_text(text):
    return ' '.join(str(text).split()).lower()

def calculate_component_similarity(model_arg, gt_arg):
    arg_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['argument']), normalize_text(gt_arg['argument'])).ratio()
    claim_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['claim']), normalize_text(gt_arg['claim'])).ratio()
    expl_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['explanation']), normalize_text(gt_arg['explanation'])).ratio()

    # ADD VERIFICATION SIMILARITY
    verif_match = 1.0 if normalize_text(model_arg['verification']) == normalize_text(gt_arg['verification']) else 0.0

    # Weight: argument=50%, claim=25%, explanation=15%, verification=10%
    return arg_sim * 0.5 + claim_sim * 0.25 + expl_sim * 0.15 + verif_match * 0.1

api = HfApi()
checkpoint_id = "llama-8B-argument_mining_V4"
base_model_id = "llama-3.1-newspaper-arguments-your_name-optimized_full"

current_total_steps = 0
try:
    commits = list(api.list_repo_commits(checkpoint_id, token=token))
    training_commits = [c for c in commits if "Training in progress" in c.title]
    current_total_steps = len(training_commits) * 4
    print(f" Resuming from step {current_total_steps}")
except:
    print("Starting fresh")

TARGET_STEPS = 600
steps_this_run = 4

if current_total_steps >= TARGET_STEPS:
    print(f"Complete! {current_total_steps}/{TARGET_STEPS}")
    exit(0)

def load_json_objects_from_txt(path):
    objs, buf, depth = [], "", 0
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    for ch in text:
        if ch == '{': depth += 1
        if depth > 0: buf += ch
        if ch == '}':
            depth -= 1
            if depth == 0 and buf.strip():
                try: objs.append(json.loads(buf))
                except: pass
                buf = ""
    return objs

raw_examples = load_json_objects_from_txt("post-training_multiarg_final.txt")
print(f"Loaded {len(raw_examples)} examples")

if len(raw_examples) > 50:
    raw_examples = raw_examples[200:250]

ids, prompts, ground_truths, articles = [], [], [], []
for idx, ex in enumerate(raw_examples):
    ex_id = ex.get("example_id", f"example_{idx:04d}")
    article = ex.get("source_text")
    gt = ex.get("ground_truth")
    if not article or not gt:
        continue
    ids.append(str(ex_id))
    ground_truths.append(gt)
    articles.append(article)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Extract argumentative units from historical text in their original form, no summaries.\\n{article}"}]
    prompts.append(messages)

print(f"Prepared {len(ids)} examples")

ds = Dataset.from_dict({
    "example_id": ids,
    "prompt": prompts,
    "ground_truth": ground_truths,
    "article": articles
})

print("\\n" + "="*80)
print("NOW INCLUDING human_verification_needed IN REWARD!")
print("="*80)
print("Similarity weights:")
print("  - argument: 50%")
print("  - claim: 25%")
print("  - explanation: 15%")
print("  - verification: 10% (NEW!)")
print("="*80 + "\\n")

reward_call_count = 0

def ground_truth_reward_func(completions, prompts=None, ground_truth=None, article=None, example_id=None, **kwargs):
    global reward_call_count
    reward_call_count += 1

    print("\\n" + "="*80)
    print(f"REWARD FUNCTION CALL #{reward_call_count}")
    print("="*80)

    num_generations = 2
    rewards = []

    if ground_truth is None:
        print("⚠️ ERROR: ground_truth parameter is None!")
        return [0.0] * len(completions)

    for i, comp in enumerate(completions):
        content = str(comp[0].get("content", "")) if isinstance(comp, (list, tuple)) else str(comp)
        ex_idx = i // num_generations

        print(f"\\n--- Completion {i+1}/{len(completions)} ---")

        if ex_idx >= len(ground_truth):
            rewards.append(0.0)
            continue

        gt = ground_truth[ex_idx]

        try:
            model_args = extract_individual_arguments(content)
            gt_args = extract_individual_arguments(gt)
        except Exception as e:
            print(f"ERROR: {e}")
            rewards.append(0.0)
            continue

        num_model, num_gt = len(model_args), len(gt_args)
        print(f"Model={num_model} args, GT={num_gt} args")

        if num_model == 0 and num_gt == 0:
            print(f"✅ BOTH_NA → reward: +0.1")
            raw_reward = 0.1
        elif num_model == 0 and num_gt > 0:
            print(f"❌ FALSE_NEGATIVE → reward: -0.15")
            raw_reward = -0.15
        elif num_model > 0 and num_gt == 0:
            print(f"❌ FALSE_POSITIVE → reward: -0.1")
            raw_reward = -0.1
        else:
            match_scores, used_model_indices = [], set()

            for gt_idx, gt_arg in enumerate(gt_args):
                best_score, best_model_idx = 0.0, None
                for model_idx, model_arg in enumerate(model_args):
                    score = calculate_component_similarity(model_arg, gt_arg)
                    if score > best_score:
                        best_score, best_model_idx = score, model_idx

                if best_score >= 0.25 and best_model_idx is not None:
                    match_scores.append(best_score)
                    used_model_indices.add(best_model_idx)

                    # Show verification match
                    model_verif = model_args[best_model_idx]['verification']
                    gt_verif = gt_arg['verification']
                    verif_match = "✓" if normalize_text(model_verif) == normalize_text(gt_verif) else "✗"
                    print(f"  ✓ Matched (score: {best_score:.2f}, verif: {verif_match})")

            matched = len(match_scores)
            missed = num_gt - matched
            extra = num_model - len(used_model_indices)

            match_reward = sum(match_scores)
            miss_penalty = missed * -0.15
            extra_penalty = extra * -0.1

            total = match_reward + miss_penalty + extra_penalty
            raw_reward = total / num_gt if num_gt > 0 else 0.0

            print(f"→ Raw reward: {raw_reward:.3f}")

        clipped_reward = max(-1.0, min(1.0, raw_reward))
        rewards.append(float(clipped_reward))

    print(f"\\nMean: {sum(rewards)/len(rewards):.3f}")
    print("="*80)
    return rewards

if current_total_steps > 0:
    print(f"\\n🔄 Loading checkpoint: {checkpoint_id}")
    model = AutoModelForCausalLM.from_pretrained(checkpoint_id, token=token, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_id, token=token)
    print(f"✅ Checkpoint loaded from step {current_total_steps}")
else:
    print(f"\\n🆕 Loading base: {base_model_id}")
    model = AutoModelForCausalLM.from_pretrained(base_model_id, token=token, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=token)
    print("✅ Base model loaded")

model.generation_config = GenerationConfig(
    bos_token_id=model.generation_config.bos_token_id,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=model.generation_config.eos_token_id if isinstance(model.generation_config.eos_token_id, int) else model.generation_config.eos_token_id[0],
    do_sample=True,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.05
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

training_args = GRPOConfig(
    output_dir="./grpo-stable-v2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    max_steps=steps_this_run,
    logging_steps=1,
    save_steps=steps_this_run,
    save_total_limit=1,
    scale_rewards="group",
    num_generations=2,
    max_completion_length=3000,
    max_prompt_length=5048,
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    seed=42,
    report_to=[],
    push_to_hub=True,
    hub_model_id=checkpoint_id,
    hub_token=token,
    optim="paged_adamw_8bit",
    generation_batch_size=2,
    generation_kwargs={
        "do_sample": True,
        "temperature": 0.1,
        "top_p": 0.95,
        "repetition_penalty": 1.05,
        "pad_token_id": tokenizer.eos_token_id
    }
)

print(f"\\n{'='*80}")
print(f"TRAINING: Steps {current_total_steps} → {current_total_steps + steps_this_run}")
print(f"NOW REWARDING CORRECT human_verification!")
print(f"{'='*80}\\n")

trainer = GRPOTrainer(
    model=model,
    reward_funcs=ground_truth_reward_func,
    args=training_args,
    train_dataset=ds,
    processing_class=tokenizer
)

trainer.train()
trainer.save_model()
trainer.push_to_hub()
print(f"\\n Complete! Progress: {current_total_steps + steps_this_run}/{TARGET_STEPS}")
"""

print(" Uploading VERIFICATION-AWARE script...")
api.upload_file(
    path_or_fileobj=grpo_training_script.encode(),
    path_in_repo="grpo-post-training/grpo_with_verification.py",
    repo_id="jobs",
    repo_type="dataset"
)
print(" Uploaded!")

print("\n Starting training WITH verification reward...\n")

!hf jobs run \
  --flavor a100-large \
  pytorch/pytorch:2.4.0-cuda12.1-cudnn9-devel \
  /bin/bash -c "\
    export HF_TOKEN='{token}' && \
    apt-get update -qq && \
    apt-get install -y -qq wget git && \
    pip install -q --upgrade pip && \
    pip install -q torch>=2.3.0 transformers accelerate datasets sentencepiece numpy bitsandbytes && \
    pip install -q git+https://github.com/huggingface/trl.git@main && \
    wget -q https://huggingface.co/datasets/jobs/resolve/main/grpo-post-training/post-training_multiarg_final.txt && \
    wget -q https://huggingface.co/datasets/jobs/resolve/main/grpo-post-training/grpo_with_verification.py && \
    python grpo_with_verification.py"


In [ ]:
from huggingface_hub import HfApi, HfFolder

api = HfApi()
token = HfFolder.get_token()

checkpoint_id = "llama-3.1-arguments-verification-v4"

grpo_training_script = """
import os
import json
import difflib
import re
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, GenerationConfig
from trl import GRPOConfig, GRPOTrainer
from huggingface_hub import login, HfApi

logging.set_verbosity_info()

token = os.environ.get('HF_TOKEN')
login(token=token)

SYSTEM_PROMPT = '''You are an expert at analyzing historical texts and you hate to summarize

OUTPUT FORMAT - EXACTLY these 4 XML tags and NOTHING else:
<argument>Original argument text OR "NA"</argument>
<claim>Core claim (implication) in one sentence OR "NA"</claim>
<explanation>Why this is an argument OR "NA"</explanation>
<human_verification_needed>True OR False</human_verification_needed>

<argument>It is reported from Malta that the British war-ships "Exmouth," "Euryalus," "Minerva," and "Sutlej" have left for Messina. The French Government has sent two armoured ships and three destroyers to Messina. President Fallieres, Premier Clemenceau, Minister Pichon, and the Presidents of the Senate and Chamber have all sent messages of sympathy to the Italian Government. The help already proffered and accepted is insufficient for the purpose. There is pressing need of extraordinary measures of help, and provisions are in great demand. There is need of doctors, tents, clothing, and provisions for the survivors, who, deprived of all necessities, are exposed to the inclemencies of the winter weather. There is need of fire engines to cope with the flames that are raging among the ruins. The railway station has collapsed. Railway carriages have been destroyed. Almost all the railway employees are dead. The streets are no longer recognisable; they look like enormous fissures in a distant and extensive heap of ruins.</argument>
<claim>Current relief efforts are inadequate and much more extensive aid is urgently needed.</claim>
<explanation>The prefect explicitly argues that existing help is "insufficient" and makes a direct claim that "extraordinary measures" are needed, presenting a clear premise-conclusion structure about the inadequacy of current response.</explanation>
<human_verification_needed>False</human_verification_needed>

EXAMPLE WITHOUT ARGUMENT:
<argument>NA</argument>
<claim>NA</claim>
<explanation>NA</explanation>
<human_verification_needed>FALSE</human_verification_needed>

RULES:
- Only output arguments that appear verbatim (or nearly verbatim) in the text
- NO SUMMARY; ONLY EXACT EXTRACTOM FROM THE TEXT; don't extract anything that is not in the text. Only extract word by word
- ONLY output these 4 XML tags
- Extract only original text without changes or use NA when you did not find an argument
- Factual reportings such as "Dem Vulkanausbruch folgten drei Sturzwellen in etwa 10 Meter Höhe" or "Almost all the inhabitants were killed; only a few thousands escaped death" are NO Arguments
- The CLAIM should say what the (implicite) argument implies, what the main conclusion is
- Give attention to implicit argumetns
- In cases of uncertainty or ambiguity, say human_verification_needed TRUE
- If no argument exists, use NA for ALL fields without explanation except <human_verification_needed>FALSE or TRUE</human_verification_needed>
- More than one argument possible for one aticle, one unit has one clear clame and all the xml structures

VERIFICATION: BEfore you print the results, double check claims and explanations of the argument. When the claim is just a translation or the explanation states that this is not an argument, dont print it'''

def extract_individual_arguments(xml_text):
    pattern = r'<argument>(.*?)</argument>\\s*<claim>(.*?)</claim>\\s*<explanation>(.*?)</explanation>\\s*<human_verification_needed>(.*?)</human_verification_needed>'
    matches = re.findall(pattern, str(xml_text), re.DOTALL | re.IGNORECASE)
    arguments = []
    for arg, claim, expl, verif in matches:
        arg_text = ' '.join(arg.strip().split())
        if arg_text.upper() == 'NA':
            continue
        arguments.append({
            "argument": arg_text,
            "claim": ' '.join(claim.strip().split()),
            "explanation": ' '.join(expl.strip().split()),
            "verification": verif.strip().lower()  # Keep this!
        })
    return arguments

def normalize_text(text):
    return ' '.join(str(text).split()).lower()

def calculate_component_similarity(model_arg, gt_arg):
    arg_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['argument']), normalize_text(gt_arg['argument'])).ratio()
    claim_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['claim']), normalize_text(gt_arg['claim'])).ratio()
    expl_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['explanation']), normalize_text(gt_arg['explanation'])).ratio()

    # ADD VERIFICATION SIMILARITY
    verif_match = 1.0 if normalize_text(model_arg['verification']) == normalize_text(gt_arg['verification']) else 0.0

    # Weight: argument=50%, claim=25%, explanation=15%, verification=10%
    return arg_sim * 0.5 + claim_sim * 0.25 + expl_sim * 0.15 + verif_match * 0.1

api = HfApi()
checkpoint_id = "llama-3.1-arguments-verification-v2"
base_model_id = "llama-3.1-newspaper-arguments-your_name-optimized_full"

current_total_steps = 0
try:
    commits = list(api.list_repo_commits(checkpoint_id, token=token))
    training_commits = [c for c in commits if "Training in progress" in c.title]
    current_total_steps = len(training_commits) * 4
    print(f" Resuming from step {current_total_steps}")
except:
    print("Starting fresh")

TARGET_STEPS = 600
steps_this_run = 4

if current_total_steps >= TARGET_STEPS:
    print(f"Complete! {current_total_steps}/{TARGET_STEPS}")
    exit(0)

def load_json_objects_from_txt(path):
    objs, buf, depth = [], "", 0
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    for ch in text:
        if ch == '{': depth += 1
        if depth > 0: buf += ch
        if ch == '}':
            depth -= 1
            if depth == 0 and buf.strip():
                try: objs.append(json.loads(buf))
                except: pass
                buf = ""
    return objs

raw_examples = load_json_objects_from_txt("post-training_multiarg_final.txt")
print(f"Loaded {len(raw_examples)} examples")

if len(raw_examples) > 50:
    raw_examples = raw_examples[:50]

ids, prompts, ground_truths, articles = [], [], [], []
for idx, ex in enumerate(raw_examples):
    ex_id = ex.get("example_id", f"example_{idx:04d}")
    article = ex.get("source_text")
    gt = ex.get("ground_truth")
    if not article or not gt:
        continue
    ids.append(str(ex_id))
    ground_truths.append(gt)
    articles.append(article)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Extract argumentative units from historical text in their original form, no summaries.\\n{article}"}]
    prompts.append(messages)

print(f"Prepared {len(ids)} examples")

ds = Dataset.from_dict({
    "example_id": ids,
    "prompt": prompts,
    "ground_truth": ground_truths,
    "article": articles
})

print("\\n" + "="*80)
print("NOW INCLUDING human_verification_needed IN REWARD!")
print("="*80)
print("Similarity weights:")
print("  - argument: 50%")
print("  - claim: 25%")
print("  - explanation: 15%")
print("  - verification: 10% (NEW!)")
print("="*80 + "\\n")

reward_call_count = 0

def ground_truth_reward_func(completions, prompts=None, ground_truth=None, article=None, example_id=None, **kwargs):
    global reward_call_count
    reward_call_count += 1

    print("\\n" + "="*80)
    print(f"REWARD FUNCTION CALL #{reward_call_count}")
    print("="*80)

    num_generations = 2
    rewards = []

    if ground_truth is None:
        print("ERROR: ground_truth parameter is None!")
        return [0.0] * len(completions)

    for i, comp in enumerate(completions):
        content = str(comp[0].get("content", "")) if isinstance(comp, (list, tuple)) else str(comp)
        ex_idx = i // num_generations

        print(f"\\n--- Completion {i+1}/{len(completions)} ---")

        if ex_idx >= len(ground_truth):
            rewards.append(0.0)
            continue

        gt = ground_truth[ex_idx]

        try:
            model_args = extract_individual_arguments(content)
            gt_args = extract_individual_arguments(gt)
        except Exception as e:
            print(f"ERROR: {e}")
            rewards.append(0.0)
            continue

        num_model, num_gt = len(model_args), len(gt_args)
        print(f"Model={num_model} args, GT={num_gt} args")

        if num_model == 0 and num_gt == 0:
            print(f"✅ BOTH_NA → reward: +0.1")
            raw_reward = 0.1
        elif num_model == 0 and num_gt > 0:
            print(f"❌ FALSE_NEGATIVE → reward: -0.15")
            raw_reward = -0.15
        elif num_model > 0 and num_gt == 0:
            print(f"❌ FALSE_POSITIVE → reward: -0.1")
            raw_reward = -0.1
        else:
            match_scores, used_model_indices = [], set()

            for gt_idx, gt_arg in enumerate(gt_args):
                best_score, best_model_idx = 0.0, None
                for model_idx, model_arg in enumerate(model_args):
                    score = calculate_component_similarity(model_arg, gt_arg)
                    if score > best_score:
                        best_score, best_model_idx = score, model_idx

                if best_score >= 0.25 and best_model_idx is not None:
                    match_scores.append(best_score)
                    used_model_indices.add(best_model_idx)

                    # Show verification match
                    model_verif = model_args[best_model_idx]['verification']
                    gt_verif = gt_arg['verification']
                    verif_match = "✓" if normalize_text(model_verif) == normalize_text(gt_verif) else "✗"
                    print(f"  ✓ Matched (score: {best_score:.2f}, verif: {verif_match})")

            matched = len(match_scores)
            missed = num_gt - matched
            extra = num_model - len(used_model_indices)

            match_reward = sum(match_scores)
            miss_penalty = missed * -0.15
            extra_penalty = extra * -0.1

            total = match_reward + miss_penalty + extra_penalty
            raw_reward = total / num_gt if num_gt > 0 else 0.0

            print(f"→ Raw reward: {raw_reward:.3f}")

        clipped_reward = max(-1.0, min(1.0, raw_reward))
        rewards.append(float(clipped_reward))

    print(f"\\nMean: {sum(rewards)/len(rewards):.3f}")
    print("="*80)
    return rewards

if current_total_steps > 0:
    print(f"\\n🔄 Loading checkpoint: {checkpoint_id}")
    model = AutoModelForCausalLM.from_pretrained(checkpoint_id, token=token, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_id, token=token)
    print(f"✅ Checkpoint loaded from step {current_total_steps}")
else:
    print(f"\\n🆕 Loading base: {base_model_id}")
    model = AutoModelForCausalLM.from_pretrained(base_model_id, token=token, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=token)
    print("✅ Base model loaded")

model.generation_config = GenerationConfig(
    bos_token_id=model.generation_config.bos_token_id,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=model.generation_config.eos_token_id if isinstance(model.generation_config.eos_token_id, int) else model.generation_config.eos_token_id[0],
    do_sample=True,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.05
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

training_args = GRPOConfig(
    output_dir="./grpo-stable-v2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    max_steps=steps_this_run,
    logging_steps=1,
    save_steps=steps_this_run,
    save_total_limit=1,
    scale_rewards="group",
    num_generations=2,
    max_completion_length=3000,
    max_prompt_length=5048,
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    seed=42,
    report_to=[],
    push_to_hub=True,
    hub_model_id=checkpoint_id,
    hub_token=token,
    optim="paged_adamw_8bit",
    generation_batch_size=2,
    generation_kwargs={
        "do_sample": True,
        "temperature": 0.1,
        "top_p": 0.95,
        "repetition_penalty": 1.05,
        "pad_token_id": tokenizer.eos_token_id
    }
)

print(f"\\n{'='*80}")
print(f"TRAINING: Steps {current_total_steps} → {current_total_steps + steps_this_run}")
print(f"NOW REWARDING CORRECT human_verification!")
print(f"{'='*80}\\n")

trainer = GRPOTrainer(
    model=model,
    reward_funcs=ground_truth_reward_func,
    args=training_args,
    train_dataset=ds,
    processing_class=tokenizer
)

trainer.train()
trainer.save_model()
trainer.push_to_hub()
print(f"\\n Complete! Progress: {current_total_steps + steps_this_run}/{TARGET_STEPS}")
"""

print(" Uploading VERIFICATION-AWARE script...")
api.upload_file(
    path_or_fileobj=grpo_training_script.encode(),
    path_in_repo="grpo-post-training/grpo_with_verification.py",
    repo_id="jobs",
    repo_type="dataset"
)
print(" Uploaded!")

print("\n Starting training WITH verification reward...\n")

!hf jobs run \
  --flavor a100-large \
  pytorch/pytorch:2.4.0-cuda12.1-cudnn9-devel \
  /bin/bash -c "\
    export HF_TOKEN='{token}' && \
    apt-get update -qq && \
    apt-get install -y -qq wget git && \
    pip install -q --upgrade pip && \
    pip install -q torch>=2.3.0 transformers accelerate datasets sentencepiece numpy bitsandbytes && \
    pip install -q git+https://github.com/huggingface/trl.git@main && \
    wget -q https://huggingface.co/datasets/jobs/resolve/main/grpo-post-training/post-training_multiarg_final.txt && \
    wget -q https://huggingface.co/datasets/jobs/resolve/main/grpo-post-training/grpo_with_verification.py && \
    python grpo_with_verification.py"



In [ ]:
from huggingface_hub import HfApi, HfFolder

api = HfApi()
token = HfFolder.get_token()

checkpoint_id = "llama-3.1-newspaper-arguments-grpo-stable-v2"

grpo_training_script = """
import os
import json
import difflib
import re
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, GenerationConfig
from trl import GRPOConfig, GRPOTrainer
from huggingface_hub import login, HfApi

logging.set_verbosity_info()

token = os.environ.get('HF_TOKEN')
login(token=token)

SYSTEM_PROMPT = '''You are an expert at analyzing historical texts and you hate to summarize

OUTPUT FORMAT - EXACTLY these 4 XML tags and NOTHING else:
<argument>Original argument text OR "NA"</argument>
<claim>Core claim (implication) in one sentence OR "NA"</claim>
<explanation>Why this is an argument OR "NA"</explanation>
<human_verification_needed>True OR False</human_verification_needed>

<argument>It is reported from Malta that the British war-ships "Exmouth," "Euryalus," "Minerva," and "Sutlej" have left for Messina. The French Government has sent two armoured ships and three destroyers to Messina. President Fallieres, Premier Clemenceau, Minister Pichon, and the Presidents of the Senate and Chamber have all sent messages of sympathy to the Italian Government. The help already proffered and accepted is insufficient for the purpose. There is pressing need of extraordinary measures of help, and provisions are in great demand. There is need of doctors, tents, clothing, and provisions for the survivors, who, deprived of all necessities, are exposed to the inclemencies of the winter weather. There is need of fire engines to cope with the flames that are raging among the ruins. The railway station has collapsed. Railway carriages have been destroyed. Almost all the railway employees are dead. The streets are no longer recognisable; they look like enormous fissures in a distant and extensive heap of ruins.</argument>
<claim>Current relief efforts are inadequate and much more extensive aid is urgently needed.</claim>
<explanation>The prefect explicitly argues that existing help is "insufficient" and makes a direct claim that "extraordinary measures" are needed, presenting a clear premise-conclusion structure about the inadequacy of current response.</explanation>
<human_verification_needed>False</human_verification_needed>

EXAMPLE WITHOUT ARGUMENT:
<argument>NA</argument>
<claim>NA</claim>
<explanation>NA</explanation>
<human_verification_needed>FALSE</human_verification_needed>

RULES:
- Only output arguments that appear verbatim (or nearly verbatim) in the text
- NO SUMMARY; ONLY EXACT EXTRACTOM FROM THE TEXT; don't extract anything that is not in the text. Only extract word by word
- ONLY output these 4 XML tags
- Extract only original text without changes or use NA when you did not find an argument
- Factual reportings such as "Dem Vulkanausbruch folgten drei Sturzwellen in etwa 10 Meter Höhe" or "Almost all the inhabitants were killed; only a few thousands escaped death" are NO Arguments
- The CLAIM should say what the (implicite) argument implies, what the main conclusion is
- Give attention to implicit argumetns
- In cases of uncertainty or ambiguity, say human_verification_needed TRUE
- If no argument exists, use NA for ALL fields without explanation except <human_verification_needed>FALSE or TRUE</human_verification_needed>
- More than one argument possible for one aticle, one unit has one clear clame and all the xml structures

VERIFICATION: BEfore you print the results, double check claims and explanations of the argument. When the claim is just a translation or the explanation states that this is not an argument, dont print it'''

def extract_individual_arguments(xml_text):
    pattern = r'<argument>(.*?)</argument>\\s*<claim>(.*?)</claim>\\s*<explanation>(.*?)</explanation>\\s*<human_verification_needed>(.*?)</human_verification_needed>'
    matches = re.findall(pattern, str(xml_text), re.DOTALL | re.IGNORECASE)
    arguments = []
    for arg, claim, expl, verif in matches:
        arg_text = ' '.join(arg.strip().split())
        if arg_text.upper() == 'NA':
            continue
        arguments.append({"argument": arg_text, "claim": ' '.join(claim.strip().split()), "explanation": ' '.join(expl.strip().split()), "verification": verif.strip().lower()})
    return arguments

def normalize_text(text):
    return ' '.join(str(text).split()).lower()

def calculate_component_similarity(model_arg, gt_arg):
    arg_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['argument']), normalize_text(gt_arg['argument'])).ratio()
    claim_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['claim']), normalize_text(gt_arg['claim'])).ratio()
    expl_sim = difflib.SequenceMatcher(None, normalize_text(model_arg['explanation']), normalize_text(gt_arg['explanation'])).ratio()
    return arg_sim * 0.5 + claim_sim * 0.3 + expl_sim * 0.2

api = HfApi()
checkpoint_id = "llama-3.1-newspaper-arguments-grpo-stable-v2"
base_model_id = "lama-3.1-newspaper-arguments-your_name-optimized_full"

# Check for checkpoint
current_total_steps = 0
try:
    commits = list(api.list_repo_commits(checkpoint_id, token=token))
    training_commits = [c for c in commits if "Training in progress" in c.title]
    current_total_steps = len(training_commits) * 4
    print(f" Found {len(training_commits)} checkpoint commits")
    print(f" Resuming from step {current_total_steps}")
except Exception as e:
    print(f"Starting fresh (no checkpoint): {e}")

TARGET_STEPS = 600
steps_this_run = 4

if current_total_steps >= TARGET_STEPS:
    print(f"🎉 Complete! {current_total_steps}/{TARGET_STEPS}")
    exit(0)

def load_json_objects_from_txt(path):
    objs, buf, depth = [], "", 0
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    for ch in text:
        if ch == '{': depth += 1
        if depth > 0: buf += ch
        if ch == '}':
            depth -= 1
            if depth == 0 and buf.strip():
                try: objs.append(json.loads(buf))
                except: pass
                buf = ""
    return objs

raw_examples = load_json_objects_from_txt("post-training_multiarg_final.txt")
print(f"Loaded {len(raw_examples)} examples")

if len(raw_examples) > 50:
    raw_examples = raw_examples[:50]

ids, prompts, ground_truths, articles = [], [], [], []
for idx, ex in enumerate(raw_examples):
    ex_id = ex.get("example_id", f"example_{idx:04d}")
    article = ex.get("source_text")
    gt = ex.get("ground_truth")
    if not article or not gt:
        continue
    ids.append(str(ex_id))
    ground_truths.append(gt)
    articles.append(article)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Extract argumentative units from historical text in their original form, no summaries.\\n{article}"}]
    prompts.append(messages)

print(f"Prepared {len(ids)} examples")

ds = Dataset.from_dict({
    "example_id": ids,
    "prompt": prompts,
    "ground_truth": ground_truths,
    "article": articles
})

reward_call_count = 0

def ground_truth_reward_func(completions, prompts=None, ground_truth=None, article=None, example_id=None, **kwargs):
    global reward_call_count
    reward_call_count += 1

    print("\\n" + "="*80)
    print(f"REWARD FUNCTION CALL #{reward_call_count}")
    print("="*80)

    num_generations = 2
    rewards = []

    if ground_truth is None:
        print(" ERROR: ground_truth parameter is None!")
        return [0.0] * len(completions)

    for i, comp in enumerate(completions):
        content = str(comp[0].get("content", "")) if isinstance(comp, (list, tuple)) else str(comp)
        ex_idx = i // num_generations

        print(f"\\n--- Completion {i+1}/{len(completions)} ---")

        if ex_idx >= len(ground_truth):
            rewards.append(0.0)
            continue

        gt = ground_truth[ex_idx]

        try:
            model_args = extract_individual_arguments(content)
            gt_args = extract_individual_arguments(gt)
        except Exception as e:
            print(f"ERROR: {e}")
            rewards.append(0.0)
            continue

        num_model, num_gt = len(model_args), len(gt_args)
        print(f"Model={num_model} args, GT={num_gt} args")

        if num_model == 0 and num_gt == 0:
            print(f" BOTH_NA → reward: +0.1")
            raw_reward = 0.1
        elif num_model == 0 and num_gt > 0:
            print(f" FALSE_NEGATIVE → reward: -0.15")
            raw_reward = -0.15
        elif num_model > 0 and num_gt == 0:
            print(f" FALSE_POSITIVE → reward: -0.1")
            raw_reward = -0.1
        else:
            match_scores, used_model_indices = [], set()

            for gt_idx, gt_arg in enumerate(gt_args):
                best_score, best_model_idx = 0.0, None
                for model_idx, model_arg in enumerate(model_args):
                    score = calculate_component_similarity(model_arg, gt_arg)
                    if score > best_score:
                        best_score, best_model_idx = score, model_idx

                if best_score >= 0.25 and best_model_idx is not None:
                    match_scores.append(best_score)
                    used_model_indices.add(best_model_idx)
                    print(f"  ✓ Matched (score: {best_score:.2f})")

            matched = len(match_scores)
            missed = num_gt - matched
            extra = num_model - len(used_model_indices)

            match_reward = sum(match_scores)
            miss_penalty = missed * -0.15
            extra_penalty = extra * -0.1

            total = match_reward + miss_penalty + extra_penalty
            raw_reward = total / num_gt if num_gt > 0 else 0.0

            print(f"→ Raw reward: {raw_reward:.3f}")

        clipped_reward = max(-1.0, min(1.0, raw_reward))
        rewards.append(float(clipped_reward))

    print(f"\\nMean: {sum(rewards)/len(rewards):.3f}")
    print("="*80)
    return rewards

# LOAD CHECKPOINT (not base!)
if current_total_steps > 0:
    print(f"\\n Loading CHECKPOINT: {checkpoint_id}")
    model = AutoModelForCausalLM.from_pretrained(checkpoint_id, token=token, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_id, token=token)
    print(f" Checkpoint loaded! Continuing from step {current_total_steps}")
else:
    print(f"\\n Loading BASE: {base_model_id}")
    model = AutoModelForCausalLM.from_pretrained(base_model_id, token=token, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=token)
    print(" Base model loaded! Starting fresh")

model.generation_config = GenerationConfig(
    bos_token_id=model.generation_config.bos_token_id,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=model.generation_config.eos_token_id if isinstance(model.generation_config.eos_token_id, int) else model.generation_config.eos_token_id[0],
    do_sample=True,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.05
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

training_args = GRPOConfig(
    output_dir="./grpo-stable-v2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    max_steps=steps_this_run,
    logging_steps=1,
    save_steps=steps_this_run,
    save_total_limit=1,
    scale_rewards="group",
    num_generations=2,
    max_completion_length=3000,
    max_prompt_length=5048,
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    seed=42,
    report_to=[],
    push_to_hub=True,
    hub_model_id=checkpoint_id,
    hub_token=token,
    optim="paged_adamw_8bit",
    generation_batch_size=2,
    generation_kwargs={
        "do_sample": True,
        "temperature": 0.1,
        "top_p": 0.95,
        "repetition_penalty": 1.05,
        "pad_token_id": tokenizer.eos_token_id
    }
)

print(f"\\n{'='*80}")
print(f"CONTINUING: Steps {current_total_steps} → {current_total_steps + steps_this_run} (of {TARGET_STEPS})")
print(f"{'='*80}\\n")

trainer = GRPOTrainer(
    model=model,
    reward_funcs=ground_truth_reward_func,
    args=training_args,
    train_dataset=ds,
    processing_class=tokenizer
)

trainer.train()
trainer.save_model()
trainer.push_to_hub()
print(f"\\n Complete! Progress: {current_total_steps + steps_this_run}/{TARGET_STEPS}")
"""

print(" Uploading CONTINUE script...")
api.upload_file(
    path_or_fileobj=grpo_training_script.encode(),
    path_in_repo="grpo-post-training/grpo_continue.py",
    repo_id="jobs",
    repo_type="dataset"
)
print(" Uploaded!")

print("\n Continuing from step 4 → 8...\n")

!hf jobs run \
  --flavor a100-large \
  pytorch/pytorch:2.4.0-cuda12.1-cudnn9-devel \
  /bin/bash -c "\
    export HF_TOKEN='{token}' && \
    apt-get update -qq && \
    apt-get install -y -qq wget git && \
    pip install -q --upgrade pip && \
    pip install -q torch>=2.3.0 transformers accelerate datasets sentencepiece numpy bitsandbytes && \
    pip install -q git+https://github.com/huggingface/trl.git@main && \
    wget -q https://huggingface.co/datasets/jobs/resolve/main/grpo-post-training/post-training_multiarg_final.txt && \
    wget -q https://huggingface.co/datasets/jobs/resolve/main/grpo-post-training/grpo_continue.py && \
    python grpo_continue.py"

print("\n This will:")
print("  1. Detect step 4 checkpoint")
print("  2. Load the checkpoint (not base)")
print("  3. Train steps 4→8")
print("  4. Save and repeat!")